# Orion

## Getting Started

### Here we'll go through a complete rocket trajectory simulation for Orion, in the case of a Nominal Flight. 
#### Let's start by importing RocketPy

In [ ]:
%pip install rocketpy
from rocketpy import *
import numpy as np

## Setting Up the Simulation

#### To run the simulations locally complete the correct location for the csv files needed (part of the file path is already written)

### Environment

In [ ]:
env = Environment(latitude=-21.90795, longitude=-48.96156, elevation=495)

In [ ]:

env.set_date(
    (2026, 9, 4, 15)
)  # Hour given in UTC time

In [ ]:
env.set_atmospheric_model(
    type="custom_atmosphere",
    pressure=None,
    temperature=None, #Leaving temperature and pressure in None means we are getting values from ISA
    wind_u=2.77778,
    wind_v=0
    )

In [ ]:
env.all_info()

### Motor

In [ ]:
from rocketpy import Fluid, CylindricalTank, MassFlowRateBasedTank, HybridMotor

In [ ]:
import csv
#insert correct location for csv files to run the code locally.
liquid_mass=f'/Entrega RocketPy - PU4/Data/liquid_mass_curve_2026__455.csv'
vapor_mass=f'/Entrega RocketPy - PU4/Data/vapor_mass_curve_2026_455.csv'

flux_time = None

with open(liquid_mass, newline='', encoding="utf-8") as f:
    reader = csv.reader(f)

    for row in reader:
        if not row:          # skip empty lines
            continue
        value = row[0].strip()
        if value == "":      # skip empty first column
            continue
        flux_time = float(value)

print(f'Flux time: {flux_time}')

In [ ]:
# Define the fluids
oxidizer_liq = Fluid(name="N2O_l", density=742.9329783371916)
oxidizer_gas = Fluid(name="N2O_g", density=188.7650429545601)

# Define tank geometry
tank_shape = CylindricalTank(154 / 2000, 0.4034) 

oxidizer_tank = MassBasedTank(
    name = "TANK",
    geometry=tank_shape,
    flux_time=(flux_time), 
    liquid=oxidizer_liq,
    liquid_mass=liquid_mass,
    gas=oxidizer_gas,
    gas_mass=vapor_mass
)

In [ ]:
Nybrid = HybridMotor(
    thrust_source="/Entrega RocketPy - PU4/Data/Nemesis_Thrust_10_07_2026.csv",
    dry_mass=0,
    dry_inertia=(0, 0, 0),
    center_of_dry_mass_position=0.813,
    grain_number=1,
    grain_separation=0,
    grain_outer_radius= 98 / 2000,
    grain_initial_inner_radius= 50 / 2000,
    grain_initial_height= 300 / 1000,
    grain_density=900,
    nozzle_radius=63.36 / 2000,
    throat_radius=26 / 2000,
    interpolation_method="linear",
    grains_center_of_mass_position=357/1000,
    reshape_thrust_curve=False,
    nozzle_position=0,
    coordinate_system_orientation="nozzle_to_combustion_chamber"
)

In [ ]:
Nybrid.add_tank(
  tank = oxidizer_tank, position = (917.5+403.4/2)/1000
)

In [ ]:
Nybrid.all_info()

### Rocket

In [ ]:
Cd_RASAero = "/Entrega RocketPy - PU4/Data/CD_RasaeroII_Orion.csv"
Orion = Rocket(
    radius= 0.0817,
    mass=33.619, 
    inertia=(25.494, 25.495, 0.161), 
    power_off_drag= Cd_RASAero,
    power_on_drag= Cd_RASAero,
    center_of_mass_without_motor=2.079, 
    coordinate_system_orientation='nose_to_tail',
    )


rail_buttons = Orion.set_rail_buttons(
    upper_button_position=1.651,
    lower_button_position=3.371,
    angular_position=45,
)

In [ ]:
Orion.add_motor(Nybrid, position=3.393) 

#### Aerodynamic Surfaces

In [ ]:
naca0012 = "/Entrega RocketPy - PU4/Data/NACA0012-radians.csv"

nose_cone = Orion.add_nose(
      length=0.58763, kind="vonKarman", position=0
    )

fin_set = Orion.add_trapezoidal_fins(
    n=4,
    root_chord=0.135, 
    tip_chord=0.130, 
    span=0.125, 
    position=3.2691, 
    cant_angle=0.5,
    airfoil=(naca0012, "degrees"),
    sweep_length=0.005,
)

tail = Orion.add_tail(
    top_radius=0.0817, bottom_radius=0.06536, length=0.0518, position=3.4244
)

In [ ]:
fin_set.draw()

#### Parachutes

In [ ]:
Main = Orion.add_parachute(
    "Main",
    cd_s=5.54, 
    trigger="apogee", 
    sampling_rate=50,
    lag=1.5,
    noise=(0, 8.3, 0.5),
)

Drogue = Orion.add_parachute(
    "Drogue",
    cd_s=1.1, 
    trigger="apogee",
    sampling_rate=50,
    lag=1.5,
    noise=(0, 8.3, 0.5),
)

#### Rocket Info

In [ ]:
Orion.all_info()

### Airbrake

#### Control Definitions

In [ ]:
import csv

# Open the CSV file for reading
filename = "/Entrega RocketPy - PU4/Data/ABLASC_2026.csv"
with open(filename, mode='r') as file:
    # Create a CSV reader with DictReader
    csv_reader = csv.reader(file)
    header = next(csv_reader)

    # Initialize an empty list to store the dictionaries
    data_list = []

    # Iterate through each row in the CSV file
    for row in csv_reader:
        # Append each row (as a dictionary) to the list
        data_list.append(row)

for i in range(len(data_list)) :
  for j in range(len(data_list[i])):
    data_list[i][j] = float(data_list[i][j])

In [ ]:
class PID:
    def __init__(self, Kp, Ki, Kd, setpoint, sample_time, min_output, max_output):
        self.Kp = Kp
        self.Ki = Ki
        self.Kd = Kd
        self.setpoint = setpoint
        self.sample_time = sample_time
        self.integral = 0
        self.derivative = 0
        self.previous_error = 0
        self.output = 0
        self.integrator_windup = 1
        self.min_output = min_output
        self.max_output = max_output

    def set_integrator_windup(self, integrator_windup):
        self.integrator_windup = integrator_windup

    def set_min_max_output(self, min_output, max_output):
        self.min_output = min_output
        self.max_output = max_output

    def update(self, current_value):
        error = self.setpoint - current_value
        self.integral += error * self.sample_time
        if self.integral > self.integrator_windup:
            self.integral = self.integrator_windup
        elif self.integral < -self.integrator_windup:
            self.integral = -self.integrator_windup
        self.derivative = (error - self.previous_error) / self.sample_time
        self.output = self.Kp * error + self.Ki * self.integral + self.Kd * self.derivative
        if self.output > self.max_output:
            self.output = self.max_output
        elif self.output < self.min_output:
            self.output = self.min_output
        self.previous_error = error
        return self.output

    def set_setpoint(self, new_setpoint):
        self.setpoint = new_setpoint

class KalmanFilter:
    def __init__(self, F, G, H, Q, R, P0, X0, process_noise):
        self.F, self.G, self.H = F, G, H
        self.Q_aux = Q
        self.Q, self.R, self.P0, self.X0 = self.Q_aux*process_noise, R, P0, X0
        self.n, self.m, self.r = F.shape[0], G.shape[1], H.shape[0]
        self.estimatesAposteriori, self.covarianceAposteriori = X0, P0
        self.estimatesApriori, self.covarianceApriori = np.zeros((self.n, 1)), np.zeros((self.n, self.n))
        self.KalmanGain, self.residual = np.zeros((self.n, self.r)), np.zeros((self.r, 1))

    def update(self, measurement):
        Sk = np.linalg.inv(self.R + self.H @ self.covarianceApriori @ self.H.T)
        self.KalmanGain = self.covarianceApriori @ self.H.T @ Sk
        self.residual = measurement - self.H @ self.estimatesApriori

        # State Update
        self.estimatesAposteriori = self.estimatesApriori + self.KalmanGain @ self.residual

        I_n = np.eye(self.n)
        KGH = I_n - self.KalmanGain @ self.H

        # Covariance update
        self.covarianceAposteriori = KGH @ self.covarianceApriori @ KGH.T + self.KalmanGain @ self.R @ self.KalmanGain.T

    def predict(self, U):
        # State Extrapolation
        self.estimatesApriori = self.F @ self.estimatesAposteriori + self.G @ U

        # Covariance Extrapolation
        self.covarianceApriori = self.F @ self.covarianceAposteriori @ self.F.T + self.Q

    def get_sigma(self):
        return np.array((np.sqrt(np.diag(self.covarianceAposteriori))))

    def get_posteriori_state(self):
        return self.estimatesAposteriori

    def get_posteriori_covariance(self):
        return self.covarianceAposteriori

    def get_priori_state(self):
        return self.estimatesApriori

    def set_process_noise(self, process_noise):
        self.Q = self.Q_aux*process_noise

    def get_Mahanalobis_Distance(self):
        return ((self.residual.T @ np.linalg.inv(self.R + self.H @ self.covarianceApriori @ self.H.T) @ self.residual)**0.5)

class RungeKutta4:
    def __init__(self, n, dt):
        self.n = n
        self.dt = dt
        self.PredictedApogee = 0
        self.u_dot = np.zeros(n)
        self.u1 = np.zeros(n)
        self.u2 = np.zeros(n)
        self.u3 = np.zeros(n)
        self.u = np.zeros(n)
        self.z = []
        self.vz = []
        self.dt_over_2 = self.dt / 2.0

    def ODE(self, t, u, input):
        self.u_dot[0] = u[1]
        self.u_dot[1] = input # TODO: Corrigir aceleração
        return self.u_dot

    def ComputeRK4(self, u0, input):
        f0 = self.ODE(self.t0, u0, input)
        f1 = self.ODE(self.t0 + self.dt_over_2, u0, input)
        f2 = self.ODE(self.t0 + self.dt_over_2, u0, input)
        f3 = self.ODE(self.t0 + self.dt, u0, input)

        for i in range(self.n):
            self.u[i] = u0[i] + self.dt * (f0[i] + 2.0 * f1[i] + 2.0 * f2[i] + f3[i]) / 6.0

        return self.u

    def predict_apogee(self, t0, stop_time, u0, input):
        max_z = u0[0]  # Initialize max_z to the first element
        self.t0 = t0

        while u0[1] >= -1e-3: # TODO: Arrumar essa porra
            z_value = u0[0]

            # Update max_z if necessary
            if z_value > max_z:
                max_z = z_value

            self.vz.append(u0[1])
            t0 += self.dt
            u0 = self.ComputeRK4(u0, input)

        self.PredictedApogee = max_z
        self.z.append(max_z)
        print("Apogeu = ", self.PredictedApogee)
        return self.PredictedApogee

class FeedforwardController:
    def __init__(self, m, S):
        self.m = m  # mass of the rocket
        self.S = S  # reference area of the rocket

    def compute_drag_coefficient(self, desired_apogee, current_altitude, current_velocity, gravity, rho):
        if (2*(desired_apogee - current_altitude)) < 1:
          divisor1 = 1
        else:
          divisor1 = (2*(desired_apogee - current_altitude))
        acc_desejado = - current_velocity**2 / divisor1
        if (self.S*rho*(current_velocity**2)) < 1:
          divisor2 = 1
        else:
          divisor2 = (self.S*rho*(current_velocity**2))
        Cd_desired = - 2*self.m*(acc_desejado - gravity)/divisor2
        if (Cd_desired < 0 or acc_desejado > 0):
          Cd_desired = 0
        return np.clip(Cd_desired, 0, 1), acc_desejado

In [ ]:
import pandas as pd

# Supondo que data_list contenha os dados do arquivo CSV
df = pd.DataFrame(data_list, columns=["deployment_level", "mach", "cd"])

def find_nearest_deployment(cd, mach):
  if cd < 0:
    return 0
  df["distance"] = (df["cd"] - cd)**2 + (df["mach"] - mach)**2
  nearest_index = df["distance"].idxmin()
  nearest_deployment = df["deployment_level"][nearest_index]
  return nearest_deployment

# Exemplo de uso
cd = 0.6
mach = 0.4
nearest_deployment = find_nearest_deployment(cd, mach)
print(f"O deployment level mais próximo para cd={cd} e mach={mach} é {nearest_deployment}")

data = pd.read_csv("/Entrega RocketPy - PU4/Data/ABLASC_2026.csv")

altitude_vector = data.iloc[:, 1]
vz_vector = data.iloc[:, 2]

def find_nearest_velocity(input_altitude):
    # Find the index of the altitude closest to the input altitude
    index = np.abs(altitude_vector - input_altitude).argmin()
    # Return the velocity at that index
    return vz_vector[index]

In [ ]:
sampling_rate = 20 # Hz
dt_k = 1/sampling_rate
noise_std = 5
process_noise = 1000

F = np.array([[1, dt_k], [0, 1]])
G = np.array([[0.5*dt_k**2], [dt_k]])
H = np.array([[1, 0]])
P0 = 10 * np.eye(2)
R = np.array([[noise_std ** 2]])
Q = np.array([[dt_k ** 4 / 4, dt_k ** 3 / 2],
              [dt_k ** 3 / 2, dt_k ** 2    ]])
x0 = np.array([[0], [0]])
KF = KalmanFilter(F, G, H, Q, R, P0, x0, process_noise=process_noise)

g = lambda h: -9.80665*(6371.009*10**3/(6371.009*10**3 + h))**2
mass = Orion.dry_mass
S =  np.pi * Orion.radius**2
FeedForward = FeedforwardController(mass, S)

Kp = 0 # 0.025
Ki = 0 # 0.075
Kd = 0 # 0.02
setpoint = 0 # TODO: Projetar a malha de controle
PID_Controller = PID(Kp, Ki, Kd, setpoint, sampling_rate, min_output = 0, max_output = 1)
desired_apogee = 1050


def controller_function(
    time, sampling_rate, state, state_history, observed_variables, air_brakes
):
    # state = [x, y, z, vx, vy, vz, e0, e1, e2, e3, wx, wy, wz]
    elevation = env.elevation
    altitude_ASL = state[2]
    altitude_AGL = altitude_ASL - elevation
    vx, vy, vz = state[3], state[4], state[5]

    speed_of_sound_real = env.speed_of_sound(altitude_ASL)
    temperature_real = env.temperature(altitude_ASL)

    # Get winds in x and y directions
    wind_x, wind_y = env.wind_velocity_x(altitude_ASL), env.wind_velocity_y(altitude_ASL)
    # Calculate Mach number
    free_stream_speed = (
        (wind_x - vx) ** 2 + (wind_y - vy) ** 2 + (vz) ** 2
    ) ** 0.5
    mach_number_real = free_stream_speed / speed_of_sound_real

    x = KF.get_posteriori_state().flatten()
    x[1] = vz
    gravity = g(x[0])
    temperature = 298.15 - 0.0065*(x[0] + elevation)
    pressure = 101325*(1 - 0.0065*(x[0] + elevation)/288.15)**(9.80665*0.02896968/(8.31447*0.0065))
    rho = pressure*0.0289644/(8.31447*temperature)
    #rho = 1.293
    #speed_of_sound = (1.4*pressure/rho)**0.5
    speed_of_sound = 331.3*(1+(temperature-273.15)/273.15)**0.5
    mach_number = x[1] / 340.29
    Cd = air_brakes.drag_coefficient(air_brakes.deployment_level, mach_number)
    drag_acceleration = -0.5*Cd*S*rho*(x[1]**2)/mass
    z_measured = altitude_AGL + np.random.normal(0,noise_std) # TODO: ver desvio do barômetro
    #z_measured = env.pressure(altitude_ASL) + np.random.normal(0,noise_std)


#aquiiiiiiiiiiiiii

    if False: #time < Nybrid.burn_out_time:
      #inputU = np.array([[0]])
      print(0)
    else:
      #KF.set_process_noise(500)
      inputU = np.array([[gravity + drag_acceleration]])
    KF.predict(inputU)
    outputY = np.array([[z_measured]])
    KF.update(outputY)
    # x = KF.get_posteriori_state().flatten()
    x[0] = altitude_AGL
    x[1] = vz
    mahanalobis_distance = KF.get_Mahanalobis_Distance()[0][0]
    predicted_Apogee = 0
    v_ideal = find_nearest_velocity(x[0])
    delta_v = v_ideal - x[1]
    if v_ideal > x[1]:
      delta_v = 0
    Cd_desired, acc_desejado = FeedForward.compute_drag_coefficient(desired_apogee, x[0], x[1], gravity, rho)
    # If below 1500 meters above ground level, air_brakes are not deployed



    if x[1] > 0.7*speed_of_sound: # or time < Nybrid.burn_out_time:
      air_brakes.deployment_level = 0
      PID_Output = 0
      exact_deployment = 0
    # Else calculate the deployment level
    else:
      PID_Output = PID_Controller.update(delta_v)
      Cd_desired = Cd_desired + PID_Output
      new_deployment_level = find_nearest_deployment(Cd_desired, mach_number)
      exact_deployment = new_deployment_level
      if Cd_desired == 0:
        new_deployment_level = 0
      if x[0] > 1050:
        new_deployment_level = 1
      # Limiting the speed of the air_brakes to 0.2 per second
      # Since this function is called every 1/sampling_rate seconds
      # the max change in deployment level per call is 0.2/sampling_rate
      max_change = 0.75 / sampling_rate
      lower_bound = air_brakes.deployment_level - max_change
      upper_bound = air_brakes.deployment_level + max_change
      new_deployment_level = min(max(new_deployment_level, lower_bound), upper_bound)

      air_brakes.deployment_level = max(0, min(new_deployment_level, 1))  # Clamp between 0 and 1

    if x[0] > desired_apogee:
      air_brakes.deployment_level = 1
    if x[1] < -1:
      air_brakes.deployment_level = 0
    # Return variables of interest to be saved in the observed_variables list
    return (
        time,
        air_brakes.deployment_level,
        air_brakes.drag_coefficient(air_brakes.deployment_level, mach_number_real),
        x,
    )

#### Adding to the Rocket

In [ ]:
import pandas as pd

# Supondo que data_list contenha os dados do arquivo CSV
df = pd.DataFrame(data_list, columns=["deployment_level", "mach", "cd"])

def find_nearest_deployment(cd, mach):
  if cd < 0:
    return 0
  df["distance"] = (df["cd"] - cd)**2 + (df["mach"] - mach)**2
  nearest_index = df["distance"].idxmin()
  nearest_deployment = df["deployment_level"][nearest_index]
  return nearest_deployment

# Exemplo de uso
cd = 0.6
mach = 0.4
nearest_deployment = find_nearest_deployment(cd, mach)
print(f"O deployment level mais próximo para cd={cd} e mach={mach} é {nearest_deployment}")

data = pd.read_csv('/Entrega RocketPy - PU4/Data/Vz_up_to_apogee.csv')

altitude_vector = data.iloc[:, 0]
vz_vector = data.iloc[:, 1]

def find_nearest_velocity(input_altitude):
    # Find the index of the altitude closest to the input altitude
    index = np.abs(altitude_vector - input_altitude).argmin()
    # Return the velocity at that index
    return vz_vector[index],
    #drag_coefficient_curve="/content/air_brakes_cd.csv",
    controller_function=controller_function,
    sampling_rate=sampling_rate,
    reference_area=np.pi * Orion.radius**2,
    clamp=True,
    initial_observed_variables=[0, 0, 0, 0, [x0.flatten()[0], x0.flatten()[1]], 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    override_rocket_drag=True,
    name="Air Brakes",


In [ ]:
air_brakes = Orion.add_air_brakes(
    drag_coefficient_curve="/Entrega RocketPy - PU4/Data/ABLASC_2026.csv",
    #drag_coefficient_curve="/content/air_brakes_cd.csv",
    controller_function=controller_function,
    sampling_rate=sampling_rate,
    reference_area=np.pi * Orion.radius**2,
    clamp=True,
    initial_observed_variables=[0, 0, 0, [x0.flatten()[0], x0.flatten()[1]]],
    override_rocket_drag=True,
    name="Air Brakes",
)

air_brakes.all_info()

## Simulating the Flight

In [ ]:
test_flight = Flight(
    rocket=Orion, environment=env, rail_length=8.2, inclination=80, heading=90
)

### Results

In [ ]:
test_flight.all_info()